# Playground Series S6E8（Predicting Smartphone Addiction）/ 「S6E8 Regime-Calibrated Rank Fusion」解説付き写し

- **コンペ**: [Predicting Smartphone Addiction — Playground Series S6E8](https://www.kaggle.com/competitions/playground-series-s6e8)（残り5日・3,019チーム）
- **元notebook**: [S6E8 Regime-Calibrated Rank Fusion | LB 0.97127](https://www.kaggle.com/code/atakanaldemir/s6e8-regime-calibrated-rank-fusion-lb-0-97127)
- **原著者**: ATAKAN ALDEMIR (atakanaldemir)
- **スコア**: Public 0.97127 / Best 0.97127（V1）・実行33秒
- **ライセンス**: Apache 2.0

> ⚠️ **これは学習目的の「解説付き写し」です。** 原著者のコードは変更していません（出力セルのみ削除）。未実行です。

## このnotebookが何をしているか（1段落の概要）

**やっていること自体は驚くほど単純**で、公開されている2つの提出ファイルを 72.5% : 27.5% の重みで
**パーセンタイル順位空間で混ぜ、もう一度順位化して提出する**、それだけです。
学習も特徴量エンジニアリングもありません。ではなぜこれを今日の教材に選んだかというと、
**「単純なブレンドを、検証可能な形で書くとこうなる」というお手本**だからです。
入力ファイルの同一性検査（ID・行順・有限性）、タイ（同値）を作らない順位化、
混ぜた結果が元からどれだけ動いたかの診断、SHA-256による出力の指紋——
このコンペの上位公開notebookの多くが「他人のsubmission.csvを3本平均して終わり」なのに対し、
この1本だけが**「混ぜた結果、何がどう変わったか」を自分で確かめている**。
学ぶべきはブレンド比ではなく、この**監査（audit）の型**です。

## 評価指標

**ROC-AUC**。「無作為に選んだ陽性1件と陰性1件を並べたとき、陽性のほうに高いスコアを付けられる確率」。

- **順位だけで決まる**: スコアを単調増加な関数で変換しても AUC は変わりません（0.2→0.9 に変えても順序が同じなら同じ値）。
- **キャリブレーションを問わない**: 「確率として正しいか」は無関係。だから確率を平均するより
  **順位を平均するほうが自然**です。片方のモデルが極端に自信過剰でも、順位に直せば影響が消えます。
- **なぜこの指標か**: 依存症ラベルのようなクラス不均衡データでは accuracy が機能せず、
  また運用時の判定閾値は後から（介入コストに応じて）決まるので、**閾値に依存しない**指標が望ましい。
- **このnotebookの設計との対応**: 融合をすべて `percentile_rank` 空間で行うのは、AUC が順位のみの指標であることへの直接的な適合です。
  さらに重みを 0.725001 と `1e-6` ずらしているのは、**有理数の重み（0.725 = 29/40）だと順位和が同点になる行が発生し、
  同点は AUC 上で「半分正解」扱いになって情報を捨てる**ため。極めて細かいが、指標の定義を理解していないと出てこない工夫です。


# S6E8 Regime-Calibrated Rank Fusion | LB 0.97127

### A small, validated diversity blend that moved the public leaderboard

This notebook combines two strong but not identical views of the S6E8 test set:

1. **72.5001%** — BYER/hboyang's 205-member rank-logit-regime fusion (**LB 0.97125**)
2. **27.4999%** — our fold-safe V13 diversity anchor (**LB 0.97124**)

The final percentile-rank blend scored **0.97127** on the public leaderboard.
The `1e-6` weight offset is intentional: it removes deterministic rational
rank-sum ties without materially changing the blend.

> There is no label or ID-based post-processing here—only aligned prediction
> ranks, explicit weights, and integrity checks.


## セル1: 環境設定と重みの宣言

**What**: ライブラリを読み込み、入出力のルートパス、ターゲット列名 `addicted_label`、
そして**ブレンド重み** `REGIME_WEIGHT = 0.725001`（残りが `ANCHOR_WEIGHT = 0.274999`）を定義します。

**Why**: 注目すべきは3点。

1. **重みが定数として先頭にある**。あとで「何対何で混ぜたのか」を探し回らずに済みます。
2. **パスを環境変数で上書き可能**にしている（`os.environ.get("S6E8_INPUT_ROOT", "/kaggle/input")`）。
   Kaggle外のローカル環境でも同じコードが動きます。
3. **`display` のフォールバック**を `try/except ImportError` で用意している。
   `IPython` が無い環境では `print` に落ちる。細かいですが「他人が動かせるコード」への配慮です。

**用語**: `rankdata`（SciPy）は配列を順位に変換する関数、`spearmanr` は**スピアマンの順位相関**
（値そのものではなく順位同士の相関）を計算します。AUCが順位の指標である以上、
モデル同士の似ている度合いも順位相関で測るのが筋が通っています。


In [ ]:
from pathlib import Path
import hashlib
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata, spearmanr

try:
    from IPython.display import display
except ImportError:  # Lightweight local reproducibility fallback.
    def display(value):
        print(value)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

INPUT_ROOT = Path(os.environ.get("S6E8_INPUT_ROOT", "/kaggle/input"))
WORKING_ROOT = Path(os.environ.get("S6E8_WORKING_ROOT", "/kaggle/working"))
TARGET = "addicted_label"
REGIME_WEIGHT = 0.725001
ANCHOR_WEIGHT = 1.0 - REGIME_WEIGHT

print(f"Input root: {INPUT_ROOT}")
print(f"Weights: regime={REGIME_WEIGHT:.6f}, anchor={ANCHOR_WEIGHT:.6f}")


## 1. Locate and validate the three aligned inputs

## セル2: 入力ファイルの所在確認と整合性チェック

**What**: `locate_unique()` で、指定したファイル名が入力ツリー内に**ちょうど1つだけ**存在することを確認します。
そのうえで、2つの予測ファイルについて
「列名が `['id','addicted_label']` か」「行数がテンプレートと一致するか」「id列が完全に一致するか」
「値に NaN や inf が無いか」を `assert` で検査します。

**Why**: ブレンドの最も多い事故は「**行の並びがずれている**」です。
2つのCSVを `pd.read_csv` して列を足し算するとき、片方がidでソートされていなければ、
**別人の予測同士を平均している**ことになります。しかもこれは例外を出さず、静かにスコアだけを壊す。
`frame["id"].equals(sample["id"])` の1行がそれを防ぎます。

`len(hits) != 1` で落とすのも同じ発想で、「同名ファイルが2つあってどちらか分からない」状態を
黙って先頭を選ぶのではなく、**エラーにする**。曖昧さを実行時に持ち込まない設計です。

**初心者向け補足**: `assert 条件, "メッセージ"` は、条件が偽なら `AssertionError` を投げて実行を止めるPythonの文です。
「ここまで来た時点で必ず成り立っているはずのこと」を書き残す道具で、
コメントと違って**嘘になったら教えてくれる**のが利点です。


In [ ]:
def locate_unique(filename, required_fragment=None):
    hits = list(INPUT_ROOT.rglob(filename))
    if required_fragment is not None:
        hits = [p for p in hits if required_fragment.lower() in str(p).lower()]
    if len(hits) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {filename!r} with fragment {required_fragment!r}; found {hits}"
        )
    return hits[0]


sample_path = locate_unique("sample_submission.csv", "playground-series-s6e8")
regime_path = locate_unique("submission_nested.csv", "rank-logit-regime")
anchor_path = locate_unique("v13_diversity_anchor_lb97124.csv")

paths = pd.DataFrame(
    {
        "role": ["competition template", "205-member regime endpoint", "V13 diversity anchor"],
        "path": [sample_path, regime_path, anchor_path],
    }
)
display(paths)

sample = pd.read_csv(sample_path)
regime = pd.read_csv(regime_path)
anchor = pd.read_csv(anchor_path)

for name, frame in {"regime": regime, "anchor": anchor}.items():
    assert list(frame.columns) == ["id", TARGET], f"Unexpected columns in {name}"
    assert len(frame) == len(sample), f"Row mismatch in {name}"
    assert frame["id"].equals(sample["id"]), f"ID/order mismatch in {name}"
    assert np.isfinite(frame[TARGET]).all(), f"Non-finite values in {name}"

print(f"[ok] {len(sample):,} aligned rows")
print("[ok] IDs and row order match the competition template")
print("[ok] Both prediction vectors are finite")


## 2. Tie-safe percentile-rank fusion

ROC-AUC depends on ordering, so percentile ranks make the two endpoints
comparable without trusting their probability calibration.

```text
final = rank(0.725001 * rank(regime) + 0.274999 * rank(anchor))
```


## セル3: タイ（同点）を作らないパーセンタイル順位融合

**What**: 各予測を `(rankdata(x, method="average") - 0.5) / n` でパーセンタイル順位（0〜1）に変換し、
重み付き和を取り、**もう一度順位化**して提出します。最後に「ユニークな予測値の数 == 行数」を assert し、
出力ファイルの SHA-256 を印字します。

**Why（なぜ2回順位化するのか）**:
1回目の順位化は「2つのモデルのスケールを揃える」ため。
2回目（`prediction = percentile_rank(blend_raw)`）は、混ぜた後の値を再び一様分布に均すためです。
AUCの値そのものは単調変換に不変なので**スコアは変わりません**が、
出力が [0,1] に均等に散らばるので、後段でさらに誰かが混ぜるときに扱いやすくなります。

**Why（`- 0.5` と `/ len` の意味）**:
順位 1..n をそのまま n で割ると最大値が 1.0（境界）になります。0.5 を引いて中央に寄せると
値域が (0, 1) の開区間に収まり、後で logit 変換（`log(p/(1-p))`）を掛けたい人が
無限大で困らずに済みます。**次に使う人のことを考えた実装**です。

**Why（タイの禁止）**:
`assert np.unique(prediction).size == len(prediction)` は「同じ値の行が1つも無いこと」を要求します。
AUC の計算では同点ペアは 0.5 点（半分正解）として扱われるので、
同点が多いと**せっかくの順序情報を捨てている**ことになります。重みを 0.725 ではなく
0.725001 にしているのは、まさにこの同点を潰すためです。

**Why（SHA-256）**: 出力ファイルのハッシュを印字しておくと、「versionを上げたのに中身が同じだった」
「別のセルが後から上書きした」といった事故を検出できます。再現性を数値として残す習慣です。

**初心者向け補足**: `rankdata(..., method="average")` は同点があったとき、
その順位を平均して割り当てます（例: 3位タイが2つなら両方 3.5）。
`method="min"` や `"dense"` など他の流儀もありますが、AUCの扱いと整合するのは `"average"` です。


In [ ]:
def percentile_rank(values):
    values = np.asarray(values, dtype=np.float64)
    return (rankdata(values, method="average") - 0.5) / len(values)


regime_rank = percentile_rank(regime[TARGET])
anchor_rank = percentile_rank(anchor[TARGET])

blend_raw = REGIME_WEIGHT * regime_rank + ANCHOR_WEIGHT * anchor_rank
prediction = percentile_rank(blend_raw)

submission = sample.copy()
submission[TARGET] = prediction

assert submission["id"].equals(sample["id"])
assert np.isfinite(prediction).all()
assert np.unique(prediction).size == len(prediction), "Unexpected rank ties"

output_path = WORKING_ROOT / "submission.csv"
submission.to_csv(output_path, index=False)

sha256 = hashlib.sha256(output_path.read_bytes()).hexdigest()
print(f"[ok] Wrote {output_path}")
print(f"[ok] Unique predictions: {np.unique(prediction).size:,}/{len(prediction):,}")
print(f"[ok] SHA-256: {sha256}")


## 3. What changed? Diversity diagnostics

## セル4: 「何が変わったのか」を測る診断

**What**: 2つの予測の**スピアマン順位相関** ρ を計算し、混ぜた結果が元の主モデルからどれだけ順位を動かしたか
（平均絶対シフト、95パーセンタイル、最大値）を表にします。さらに散布図とヒストグラムを描きます。

**Why**: これがこのnotebookで一番学ぶ価値のある部分です。

アンサンブルが効くための条件は「**個々が十分強く、かつ互いに間違い方が違う**」ことです。
ρ が 1.0 に近ければ2つは実質同じモデルで、混ぜても何も起きません（＝スコアは公開LBのノイズ分しか動かない）。
ρ が低すぎれば片方が弱いだけかもしれない。**混ぜる前に相関を見る**のは、
「効いたかどうか」をLBスコアという1個の数字だけで判断しないための最低限の作法です。

散布図（横軸=主モデルの順位、縦軸=アンカーの順位、色=乖離幅）は、
**どの順位帯で2つが食い違っているか**を見るための図です。対角線から離れた点が「議論のある行」。
ヒストグラム（順位シフト×1000）は、補正が意図通り「ほとんどの行では小さく、一部でだけ効いている」かを確認します。

**注意して読むべき点**: この診断は**テストデータ上の予測同士**を比べているだけで、
**正解ラベルを使った検証ではありません**。つまり「2つは違う」ことは分かっても
「混ぜたほうが良い」ことの証明にはなっていません。原著者もそれは分かっていて、
セル8の解釈では "the kind of diversity that **can** help"（助けになり**得る**）と慎重な言い方をしています。
公開LB 0.97125 → 0.97127 という +0.00002 の差が、
private split で残るかどうかは全く別の話です（このコンペでは「0.97110超はすべて過学習」と主張する
[別notebook](https://www.kaggle.com/code/szymonkapiski/why-every-s6e8-notebook-above-0-97110-overfits) もあります）。


In [ ]:
rho = float(spearmanr(regime_rank, anchor_rank).statistic)
shift_from_regime = prediction - regime_rank

summary = pd.DataFrame(
    {
        "diagnostic": [
            "Spearman(regime, anchor)",
            "Mean absolute rank shift vs regime",
            "95th percentile |rank shift|",
            "Maximum |rank shift|",
            "Final unique ranks",
        ],
        "value": [
            f"{rho:.6f}",
            f"{np.mean(np.abs(shift_from_regime)):.6f}",
            f"{np.quantile(np.abs(shift_from_regime), 0.95):.6f}",
            f"{np.max(np.abs(shift_from_regime)):.6f}",
            f"{np.unique(prediction).size:,}",
        ],
    }
)
display(summary.style.hide(axis="index"))

rng = np.random.default_rng(42)
idx = rng.choice(len(prediction), size=min(12_000, len(prediction)), replace=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))

axes[0].scatter(
    regime_rank[idx], anchor_rank[idx],
    c=np.abs(anchor_rank[idx] - regime_rank[idx]),
    cmap="viridis", s=8, alpha=0.45, linewidths=0,
)
axes[0].plot([0, 1], [0, 1], "--", color="crimson", lw=1.2)
axes[0].set(
    title=f"Complementary ordering (Spearman = {rho:.5f})",
    xlabel="205-member regime rank",
    ylabel="V13 diversity-anchor rank",
)

axes[1].hist(shift_from_regime * 1_000, bins=90, color="#35618f", alpha=0.9)
axes[1].axvline(0, ls="--", color="crimson", lw=1.2)
axes[1].set(
    title="Final correction remains deliberately small",
    xlabel="Rank shift vs regime endpoint (×1000)",
    ylabel="Rows",
)

plt.tight_layout()
plt.show()


### Interpretation

The endpoints are strongly correlated, as expected, but not identical. The
anchor contributes a small ordering correction rather than replacing the
regime model. That is exactly the kind of diversity that can help an AUC blend:
most rows barely move, while disagreement regions receive a controlled nudge.


## 4. Submission preview

## セル5: 提出プレビュー

**What**: 提出データの先頭10行と、予測値の基本統計量（`describe()`）を表示します。

**Why**: `describe()` で min/max/mean を見れば、順位化が効いていれば
mean ≈ 0.5、min ≈ 0、max ≈ 1 の一様分布になっているはずです。
**「期待した形になっているか」を目視で1秒確認する**ための、安価で有効な最後のチェックです。
提出前にこれを見る習慣があるだけで、列名間違いや全行同値といった事故はほぼ防げます。


In [ ]:
display(submission.head(10))
display(submission[TARGET].describe().to_frame("final prediction"))


## Credits and reproducibility

The strongest endpoint is **BYER/hboyang's open 205-member rank-logit-regime
fusion**. Please visit and credit the original work:

- [S6E8 Rank-Logit-Regime Fusion | LB 0.97125](https://www.kaggle.com/code/hboyang/s6e8-rank-logit-regime-fusion-lb0-97125)

The diversity-anchor campaign was inspired by **Szymon Kapiński's** counter-
intuitive weak-model research and public CC0 artifacts:

- [S6E8 — The score I would not pick](https://www.kaggle.com/code/szymonkapiski/s6e8-the-score-i-would-not-pick)
- [S6E8 50 weakest OOF models](https://www.kaggle.com/datasets/szymonkapiski/s6e8-50-weakest-oof-models)

Thanks also to Naji, boltuzamaki, Ray Kretzschmar, Darius Hafshar, Adarsh and
the wider S6E8 community for releasing aligned OOF libraries.

The output above is deterministic. If this compact, audited blend helped your
experiments, an upvote is always appreciated. Good luck! 🚀
